# Protein Surprisal Atlas — Full Pilot Run (GPU / Colab)

This notebook runs the **authoritative pilot** for `glenritschel/protein-surprisal-atlas` end-to-end on a Colab GPU:

1. Download reviewed human proteome (UniProt) + QC
2. Build the stratified 500-protein pilot (+ UniRef family sizes)
3. Score all 500 with **sampled-mask** (fast) surprisal
4. Score the validation subset with **exact** masked-marginal surprisal
5. Validate approximate vs exact (the **Spearman ≥ 0.90 gate**)
6. Run the analysis (figures 1–7, 12–13 + tables)
7. Supplementary **per-residue** family-size model (the meaningful confound test)
8. Zip and download all results

> **Before running:** set **Runtime → Change runtime type → GPU** (T4 is fine).
> Full 500/50 run is roughly 20–40 min on a T4. To smoke-test first, set `PILOT_SIZE=30`, `EXACT_SIZE=15` in the parameters cell.

In [ ]:
# --- GPU check ---
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Set Runtime -> Change runtime type -> GPU, then rerun.")
!nvidia-smi -L

## 1. Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/glenritschel/protein-surprisal-atlas.git"
BRANCH   = "main"          # change to a feature branch if you want
REPO_DIR = "/content/protein-surprisal-atlas"

if os.path.isdir(REPO_DIR):
    print("Repo exists; pulling latest...")
    !cd {REPO_DIR} && git fetch --quiet origin {BRANCH} && git checkout --quiet {BRANCH} && git pull --quiet
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -1

## 2. Install dependencies
Colab already ships a CUDA build of PyTorch, so we install everything **except** torch to avoid clobbering GPU support.

In [ ]:
!pip install -q transformers pyarrow statsmodels biopython requests tqdm pyyaml scipy scikit-learn matplotlib pydantic
import transformers, statsmodels, pyarrow
print("transformers", transformers.__version__, "| statsmodels", statsmodels.__version__)
%env TOKENIZERS_PARALLELISM=false

## 3. Parameters & config

We write a `colab_config.yaml` derived from the repo's `config.yaml`, overriding pilot sizes, device, and a larger exact batch size for the GPU. Leave `PILOT_SIZE=500`, `EXACT_SIZE=50` for the real run.

In [ ]:
import yaml

PILOT_SIZE = 500     # set to 30 for a quick smoke test
EXACT_SIZE = 50      # set to 15 for a quick smoke test
SEED       = 42

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["project"]["seed"] = SEED
cfg["model"]["device"] = "auto"           # picks cuda when available
cfg["model"]["scoring_method"] = "sampled_mask"
cfg.setdefault("scoring", {})["exact_batch_size"] = 64   # bigger batches on GPU
cfg["scoring"]["sampled_mask_passes"] = 7
cfg["pilot"]["target_size"] = PILOT_SIZE
cfg["pilot"]["exact_validation_size"] = EXACT_SIZE

with open("colab_config.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(open("colab_config.yaml").read())

# Ensure repo root is importable by the scripts (they do `from src.protein_atlas...`)
os.environ["PYTHONPATH"] = os.getcwd()
print("PYTHONPATH =", os.environ["PYTHONPATH"])

## 4. Download proteome + QC
Downloads the reviewed human proteome (~20k entries, a few MB) and runs sequence QC.

In [ ]:
!PYTHONPATH=$PWD python scripts/download_data.py --config colab_config.yaml
import json
print(json.dumps(json.load(open("data/interim/qc_report.json")), indent=2))

## 5. Build the pilot + family sizes

Selects the stratified pilot and retrieves UniRef cluster sizes from UniProt (≈ `PILOT_SIZE`×2 REST calls; a few minutes for 500). We then sanity-check that `log_family_size` is populated (not all-NaN).

In [ ]:
!PYTHONPATH=$PWD python scripts/build_pilot.py --config colab_config.yaml

import pandas as pd
pilot = pd.read_parquet("data/interim/pilot_set.parquet")
print("Pilot proteins:", len(pilot))
print("Family-size NaN count:", pilot["log_family_size"].isna().sum(), "/", len(pilot))
print(pilot["uniref50_cluster_size"].describe())
pilot[["uniprot_id","gene_symbol","sequence_length","uniref50_cluster_size","log_family_size"]].head()

## 6. Score the full pilot (sampled-mask)
~500 proteins. Checkpointed, so it resumes if interrupted.

In [ ]:
!PYTHONPATH=$PWD python scripts/score_pilot.py --config colab_config.yaml --method sampled_mask

## 7. Score the validation subset (exact masked-marginal)
This is the slow one on CPU; on GPU the ~50-protein subset should complete. Verify the row count matches `EXACT_SIZE`.

In [ ]:
!PYTHONPATH=$PWD python scripts/score_pilot.py --config colab_config.yaml --method exact --validation-only

import pandas as pd
ex = pd.read_parquet("results/tables/validation_protein_scores_exact.parquet")
print("Exact proteins scored:", len(ex), "(target:", EXACT_SIZE, ")")
if len(ex) < EXACT_SIZE:
    print("WARNING: exact run did not cover the full validation subset — the Spearman gate below will be weak.")

## 8. Validate approximate vs exact — the Spearman ≥ 0.90 gate

In [ ]:
!PYTHONPATH=$PWD python scripts/validate_scoring.py --config colab_config.yaml

import pandas as pd
m = pd.read_csv("results/reports/validation_metrics.csv")
display(m)
rho = float(m["spearman_rho"].iloc[0]); n = int(m["n_proteins"].iloc[0])
print(f"\nSpearman rho = {rho:.4f} on n = {n} proteins -> "
      + ("PASS" if (rho >= 0.90 and n >= 20) else "NOT yet a valid gate (need rho>=0.90 on a real n)"))

## 9. Run the analysis (tables + figures 1–7, 12–13)

In [ ]:
!PYTHONPATH=$PWD python scripts/run_analysis.py --config colab_config.yaml
import json
print(json.dumps(json.load(open("results/reports/regression_results.json")), indent=2))

## 10. Supplementary: the *meaningful* family-size model (per-residue)

The repo's `statistics.py` regresses **total** surprisal on length, where length trivially explains ~94% and family size looks negligible by construction. The informative test is on **bits per residue**, controlling for length, with cluster-robust SEs on UniRef50 cluster. This cell computes that.

In [ ]:
import pandas as pd, numpy as np
import statsmodels.formula.api as smf
from scipy.stats import spearmanr

df = pd.read_parquet("results/tables/pilot_protein_scores_sampled_mask.parquet")
df = df.dropna(subset=["bits_per_residue", "sequence_length", "log_family_size"]).copy()
print(f"n = {len(df)} proteins with complete data\n")

m0 = smf.ols("bits_per_residue ~ sequence_length", data=df).fit()
m1 = smf.ols("bits_per_residue ~ sequence_length + log_family_size", data=df).fit()
m1c = smf.ols("bits_per_residue ~ sequence_length + log_family_size",
              data=df).fit(cov_type="cluster", cov_kwds={"groups": df["uniref50_cluster_id"]})

print("Per-residue surprisal model (bits/residue):")
print(f"  R^2 length-only         : {m0.rsquared:.4f}")
print(f"  R^2 length + family     : {m1.rsquared:.4f}")
print(f"  delta R^2 from family   : {m1.rsquared - m0.rsquared:.4f}")
print(f"  family coef (bits/res)  : {m1.params['log_family_size']:+.4f} per log-member")
print(f"  family p (OLS)          : {m1.pvalues['log_family_size']:.2e}")
print(f"  family p (cluster-robust): {m1c.pvalues['log_family_size']:.2e}")

rho, p = spearmanr(df["bits_per_residue"], df["log_family_size"])
print(f"\n  Spearman(bits/res, log_family_size) = {rho:+.3f}  (p = {p:.2e})")
print("\nFamily-size spread in this pilot (UniRef50 members):")
print(df["uniref50_cluster_size"].describe())

## 11. Control sanity check
Natural sequences should be **less** surprising than shuffled / reversed / random controls.

In [ ]:
import pandas as pd
c = pd.read_csv("results/tables/control_sequence_comparisons.csv")
summary = c.groupby("type")["total_surprisal_bits"].mean().sort_values()
print("Mean total surprisal by control type (lower = more predictable):")
print(summary.to_string())
print("\nOK if 'natural' is the smallest." )

## 12. View key figures inline

In [ ]:
from IPython.display import Image, display
import os
for fn in ["fig1_bits_per_residue_hist.png",
           "fig6_scoring_validation.png",
           "fig5_controls.png",
           "fig7_confound.png"]:
    p = os.path.join("results/figures", fn)
    if os.path.exists(p):
        print(fn); display(Image(filename=p, width=680))
    else:
        print("missing:", fn)

## 13. Download all results

Zips `results/` (figures, tables, reports) and downloads it. To keep them permanently, uncomment the Google Drive lines.

In [ ]:
import shutil
shutil.make_archive("psa_pilot_results", "zip", "results")
print("Zipped:", os.path.getsize("psa_pilot_results.zip")//1024, "KB")

from google.colab import files
files.download("psa_pilot_results.zip")

# --- optional: save to Google Drive instead ---
# from google.colab import drive; drive.mount('/content/drive')
# shutil.copy('psa_pilot_results.zip', '/content/drive/MyDrive/psa_pilot_results.zip')